# Modul 17: PyTorch-Tensoren, DataLoader und dichte Netze

    **Notebooktyp:** Übungs- und Bewertungsnotebook  
    **Vorlesungen dieses Moduls:** Tensoren und Loader, Dichtes PyTorch-Netz  
    **Erwarteter Schwierigkeitsgrad:** Mittlere bis fortgeschrittene PyTorch-Anwendung  
    **Orientierungszeit:** etwa 130 bis 180 Minuten

    ## Überblick

    Sie untersuchen Tensoren und Autograd, organisieren Daten mit TensorDataset und DataLoader und schreiben anschließend ein vollständiges Training für ein kleines dichtes PyTorch-Netz. Metriken, Eval-Modus und reproduzierbares Speichern schließen den Workflow ab.

    ## Verwendete Vorlesungsnotebooks

    Die Aufgaben wurden aus dem Inhalt beider Vorlesungen dieses Moduls abgeleitet:

    - `ML Für Anfänger - Record_Module_17A_20260723.ipynb`
- `ML Für Anfänger - Record_Module_17B_20260723.ipynb`

    ## Colab-Kompatibilität

    Dieses Notebook ist für die kostenlose Version von Google Colab ausgelegt. Die Daten sind eingebaut, synthetisch erzeugt oder öffentlich verfügbar. Modelle und Trainingsbudgets sind bewusst klein gehalten. Führen Sie die Zellen in der vorgegebenen Reihenfolge aus.

## Lernziele

    Nach der Bearbeitung sollen Sie:

    - PyTorch-Tensoren, Formen, Datentypen, NumPy-Konvertierung und Broadcasting sicher verwenden.
- Automatische Gradienten mit Autograd berechnen und kontrollieren.
- Trainings-, Validierungs- und Testdaten mit TensorDataset und DataLoader organisieren.
- Ein eigenes nn.Module mit einer korrekten forward-Methode definieren.
- Loss, Optimierer sowie train- und eval-Modus in einer Trainingsschleife korrekt einsetzen.
- Metriken visualisieren und Modellgewichte reproduzierbar speichern und laden.

    ## Bewertete Fähigkeiten

    - torch.Tensor, dtype, device, NumPy-Konvertierung und Broadcasting
- requires_grad, backward und grad
- TensorDataset und DataLoader
- nn.Module, forward und BCEWithLogitsLoss
- Training, Validierung, eval, no_grad und state_dict

## Arbeitsanweisungen

Bearbeiten Sie die Aufgaben in der angegebenen Reihenfolge. Schreiben Sie Ihren Code ausschließlich in die klar markierten Arbeitszellen. Ergänzen Sie nach jeder Aufgabe eine kurze fachliche Reflexion. Verwenden Sie das Testset nicht für Modellwahl oder Hyperparameterentscheidungen, sofern die Aufgabe dies nicht ausdrücklich als abschließenden Schritt verlangt.

- Führen Sie zuerst das gemeinsame Setup aus.
- Verändern Sie vorgegebene Splits und Seeds nur, wenn eine Aufgabe dies ausdrücklich erlaubt.
- Prüfen Sie Formen, Datentypen und Wertebereiche frühzeitig.
- Begründen Sie Modell-, Metrik- und Visualisierungsentscheidungen.
- Achten Sie auf Datenleckage und eine saubere Trennung von Training, Validierung und Test.

## Gemeinsames Setup

Führen Sie diese Zelle einmal aus, bevor Sie mit Aufgabe 1 beginnen.

In [ ]:
# Gemeinsames PyTorch-Setup
import io
import os
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset

RANDOM_SEED = 42
FAST_MODE = os.environ.get("COURSE_FAST", "0") == "1"
OFFLINE_MODE = os.environ.get("COURSE_OFFLINE", "0") == "1"

np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
torch.use_deterministic_algorithms(False)
warnings.filterwarnings("ignore", category=FutureWarning)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

from sklearn.datasets import make_moons
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Ein nichtlinearer Zweiklassendatensatz wird lokal erzeugt.
X_17, y_17 = make_moons(n_samples=720, noise=0.24, random_state=RANDOM_SEED)
X_train_valid_17, X_test_17, y_train_valid_17, y_test_17 = train_test_split(
    X_17,
    y_17,
    test_size=0.20,
    stratify=y_17,
    random_state=RANDOM_SEED,
)
X_train_17, X_valid_17, y_train_17, y_valid_17 = train_test_split(
    X_train_valid_17,
    y_train_valid_17,
    test_size=0.25,
    stratify=y_train_valid_17,
    random_state=RANDOM_SEED,
)

scaler_17 = StandardScaler()
X_train_17 = scaler_17.fit_transform(X_train_17).astype("float32")
X_valid_17 = scaler_17.transform(X_valid_17).astype("float32")
X_test_17 = scaler_17.transform(X_test_17).astype("float32")
y_train_17 = y_train_17.astype("float32")
y_valid_17 = y_valid_17.astype("float32")
y_test_17 = y_test_17.astype("float32")

print("Train/Valid/Test:", X_train_17.shape, X_valid_17.shape, X_test_17.shape)

print("PyTorch-Version:", torch.__version__)
print("Gerät:", DEVICE)


## Aufgabe 1: Tensoren, Broadcasting und Autograd

    Untersuchen Sie grundlegende PyTorch-Tensoroperationen.

1. Erzeugen Sie einen Skalar, Vektor, eine Matrix und einen kleinen Bildstapel mit ausdrücklich gewählten Datentypen.
2. Geben Sie Form, Dimensionen, Datentyp und Gerät aus.
3. Konvertieren Sie eine NumPy-Matrix zu PyTorch und wieder zurück. Vermeiden Sie eine unbeabsichtigte gemeinsame Speicheränderung.
4. Standardisieren Sie eine `3 x 2`-Matrix durch Broadcasting.
5. Berechnen Sie für `loss(w) = mean((w*x-y)^2)` den Gradienten mit `backward()` und vergleichen Sie ihn mit der manuellen Ableitung.

> **Hinweis:** Prüfen Sie bei jedem Tensor sowohl Form als auch Datentyp und Gerät.

In [ ]:
feature_matrix_17 = torch.tensor(
    [[2.0, 10.0], [4.0, 14.0], [6.0, 18.0]],
    dtype=torch.float32,
)
means_17 = torch.tensor([4.0, 14.0], dtype=torch.float32)
stds_17 = torch.tensor([2.0, 4.0], dtype=torch.float32)

# ============================================================


In [ ]:
# ============================================================
# IHR CODE HIER / YOUR CODE HERE
# ============================================================

# Schreiben Sie Ihre vollständige Lösung in diese Zelle.


### Reflexion zu Aufgabe 1

    > **Ihre Antwort:**  
    > Beschreiben Sie kurz Ihre Beobachtungen, begründen Sie wichtige Entscheidungen und nennen Sie mindestens eine mögliche Fehlerquelle.

**Pädagogischer Hinweis:** Prüfen Sie bei jedem Tensor sowohl Form als auch Datentyp und Gerät.

## Aufgabe 2: TensorDataset und DataLoader korrekt aufteilen

    Organisieren Sie die vorbereiteten Daten für das Training.

1. Wandeln Sie Merkmale und Labels in `float32`-Tensoren um. Labels sollen die Form `(N, 1)` besitzen.
2. Erstellen Sie getrennte `TensorDataset`-Objekte für Training, Validierung und Test.
3. Erstellen Sie DataLoader mit Batchgröße 32. Mischen Sie nur das Training.
4. Verwenden Sie einen festen Generator für reproduzierbares Shuffling.
5. Untersuchen Sie einen Batch und prüfen Sie Formen, Datentypen und Beispielzahl.
6. Erklären Sie, warum der Test-Loader nicht zum Modell- oder Schwellenwertvergleich verwendet werden darf.

> **Hinweis:** Richten Sie die Labelform bereits im Dataset an der späteren Loss-Funktion aus.

In [ ]:
batch_size_17 = 32

# Speichern Sie die Loader als train_loader_17, valid_loader_17 und
# test_loader_17, damit spätere Aufgaben sie verwenden können.

# ============================================================


In [ ]:
# ============================================================
# IHR CODE HIER / YOUR CODE HERE
# ============================================================

# Schreiben Sie Ihre vollständige Lösung in diese Zelle.


### Reflexion zu Aufgabe 2

    > **Ihre Antwort:**  
    > Beschreiben Sie kurz Ihre Beobachtungen, begründen Sie wichtige Entscheidungen und nennen Sie mindestens eine mögliche Fehlerquelle.

**Pädagogischer Hinweis:** Richten Sie die Labelform bereits im Dataset an der späteren Loss-Funktion aus.

## Aufgabe 3: Ein eigenes nn.Module definieren

    Implementieren Sie ein dichtes binäres Netz als eigene PyTorch-Klasse.

1. Erben Sie von `nn.Module`.
2. Definieren Sie eine Architektur `2 -> 24 -> 12 -> 1` mit ReLU in den verborgenen Schichten.
3. Geben Sie in `forward` **Logits** und keine Sigmoid-Wahrscheinlichkeiten zurück.
4. Verschieben Sie das Modell auf `DEVICE`.
5. Richten Sie `BCEWithLogitsLoss` und Adam ein.
6. Prüfen Sie die Ausgabeform eines Batches und zählen Sie trainierbare Parameter.

> **Hinweis:** Kombinieren Sie Sigmoid nicht zusätzlich mit `BCEWithLogitsLoss`.

In [ ]:
# Speichern Sie Modell, Loss und Optimierer als model_17,
# criterion_17 und optimizer_17.

# ============================================================


In [ ]:
# ============================================================
# IHR CODE HIER / YOUR CODE HERE
# ============================================================

# Schreiben Sie Ihre vollständige Lösung in diese Zelle.


### Reflexion zu Aufgabe 3

    > **Ihre Antwort:**  
    > Beschreiben Sie kurz Ihre Beobachtungen, begründen Sie wichtige Entscheidungen und nennen Sie mindestens eine mögliche Fehlerquelle.

**Pädagogischer Hinweis:** Kombinieren Sie Sigmoid nicht zusätzlich mit `BCEWithLogitsLoss`.

## Aufgabe 4: Trainings- und Evaluationsschleifen schreiben

    Trainieren Sie `model_17` mit klar getrennten Trainings- und Evaluationsphasen.

1. Schreiben Sie eine Funktion für eine Trainingsepoche mit `model.train()`, `zero_grad()`, Vorwärtslauf, Loss, `backward()` und `step()`.
2. Schreiben Sie eine Evaluationsfunktion mit `model.eval()` und `torch.no_grad()`.
3. Speichern Sie pro Epoche Loss und Accuracy für Training und Validierung.
4. Implementieren Sie einfaches Early Stopping anhand des Validierungsverlusts und sichern Sie die besten Gewichte im Speicher.
5. Stellen Sie die besten Gewichte wieder her und bewerten Sie das Testset genau einmal.

> **Hinweis:** Multiplizieren Sie den Batch-Loss mit der Batchgröße, bevor Sie über unterschiedlich große Batches mitteln.

In [ ]:
max_epochs_17 = 10 if FAST_MODE else 80
patience_17 = 10

# ============================================================


In [ ]:
# ============================================================
# IHR CODE HIER / YOUR CODE HERE
# ============================================================

# Schreiben Sie Ihre vollständige Lösung in diese Zelle.


### Reflexion zu Aufgabe 4

    > **Ihre Antwort:**  
    > Beschreiben Sie kurz Ihre Beobachtungen, begründen Sie wichtige Entscheidungen und nennen Sie mindestens eine mögliche Fehlerquelle.

**Pädagogischer Hinweis:** Multiplizieren Sie den Batch-Loss mit der Batchgröße, bevor Sie über unterschiedlich große Batches mitteln.

## Aufgabe 5: Integration: Metriken visualisieren und Gewichte sicher laden

    Schließen Sie das Mini-Projekt mit Diagnose und Reproduzierbarkeit ab.

1. Stellen Sie Loss und Accuracy für Training und Validierung dar.
2. Erstellen Sie eine Konfusionsmatrix für das Testset und zeigen Sie fünf besonders unsichere Beispiele.
3. Speichern Sie `state_dict`, Architekturmetadaten, Schwellenwert und Seed in einem In-Memory-Artefakt.
4. Erzeugen Sie eine neue Modellinstanz, laden Sie den Zustand mit `map_location` und versetzen Sie sie in den Eval-Modus.
5. Bestätigen Sie, dass die neue Instanz für zwölf Referenzbeispiele dieselben Logits und Klassen liefert.
6. Erläutern Sie, warum ein `state_dict` ohne Architektur- und Vorverarbeitungsinformationen kein vollständiges Produktionsartefakt ist.

> **Hinweis:** Erstellen Sie die neue Modellinstanz mit exakt derselben Architektur, bevor Sie den Zustand laden.

In [ ]:
# ============================================================


In [ ]:
# ============================================================
# IHR CODE HIER / YOUR CODE HERE
# ============================================================

# Schreiben Sie Ihre vollständige Lösung in diese Zelle.


### Reflexion zu Aufgabe 5

    > **Ihre Antwort:**  
    > Beschreiben Sie kurz Ihre Beobachtungen, begründen Sie wichtige Entscheidungen und nennen Sie mindestens eine mögliche Fehlerquelle.

**Pädagogischer Hinweis:** Erstellen Sie die neue Modellinstanz mit exakt derselben Architektur, bevor Sie den Zustand laden.

## Abschluss und Selbstkontrolle

Prüfen Sie vor der Abgabe, ob alle Arbeitszellen ausgefüllt sind, das Notebook von oben nach unten ohne unerwartete Fehler läuft, alle Diagramme beschriftet sind und jede Reflexion Ihre Beobachtungen sowie mindestens eine mögliche Fehlerquelle enthält.

- Alle Aufgaben und Unterpunkte wurden bearbeitet.
- Verwendete Seeds und Datenpartitionen sind nachvollziehbar.
- Testdaten wurden nicht vorzeitig für Entscheidungen genutzt.
- Ergebnisse werden vorsichtig und fachlich begründet interpretiert.
- Es gibt keine hardcodierten lokalen Dateipfade oder privaten Zugangsdaten.